<a href="https://colab.research.google.com/github/knavya8279-jpg/KUMARI-NAVYA-2026B03110172/blob/main/KUMARI_NAVYA(2026B03110172).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
# PROJECT 22: RESTAURANT ORDERS & MENU PERFORMANCE ANALYSIS

In [40]:
import pandas as pd
import numpy as np

In [41]:
# 1. LOAD AND INSPECT DATASET
df = pd.read_csv("22_restaurant_orders.csv")

print("\n--- FIRST 5 RECORDS ---")
print(df.head())

print("\n--- DATASET INFORMATION ---")
print(df.info())

print("\n--- DATASET SHAPE ---")
print(df.shape)

print("\n--- STATISTICAL SUMMARY ---")
print(df.describe(include="all"))




--- FIRST 5 RECORDS ---
  Order_ID Customer_ID          Item  Item_Quantity  Item_Price     Category  \
0   O00337       C0050       Brownie            1.0       110.0     Desserts   
1   O00332       C0148       Biryani            1.0       220.0  Main Course   
2   O00368       C0235         Pasta            2.0       180.0  Main Course   
3   O00051       C0213  French Fries            1.0        80.0       Snacks   
4   O00003       C0161         Pizza            1.0       250.0    Fast Food   

  Order_Type       Order_DateTime  
0   Takeaway  2026-02-15 07:36:00  
1   Takeaway  2026-02-06 18:29:00  
2   Takeaway  2026-03-07 12:40:00  
3   Takeaway  2026-03-26 17:14:00  
4   Delivery  2026-01-10 16:22:00  

--- DATASET INFORMATION ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1116 entries, 0 to 1115
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        1116 non-null   object 
 1   Cus

In [42]:
# 2. CHECK MISSING VALUES AND DUPLICATE ORDER_ID
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- DUPLICATE ORDER_ID RECORDS ---")
print(df[df.duplicated(subset="Order_ID", keep=False)])

print("\nTotal Duplicate Order_ID Records:",
      df.duplicated(subset="Order_ID").sum())


--- MISSING VALUES ---
Order_ID           0
Customer_ID        0
Item               0
Item_Quantity      8
Item_Price        10
Category           0
Order_Type         0
Order_DateTime     0
dtype: int64

--- DUPLICATE ORDER_ID RECORDS ---
     Order_ID Customer_ID              Item  Item_Quantity  Item_Price  \
0      O00337       C0050           Brownie            1.0       110.0   
1      O00332       C0148           Biryani            1.0       220.0   
2      O00368       C0235             Pasta            2.0       180.0   
3      O00051       C0213      French Fries            1.0        80.0   
4      O00003       C0161             Pizza            1.0       250.0   
...       ...         ...               ...            ...         ...   
1111   O00136       C0184      French Fries            3.0        80.0   
1112   O00362       C0216           Biryani            1.0       220.0   
1113   O00308       C0002             Pasta            1.0       180.0   
1114   O00052      

In [43]:
# 3. CLEAN MISSING ITEM_QUANTITY AND ITEM_PRICE
df["Item_Quantity"] = pd.to_numeric(
    df["Item_Quantity"], errors="coerce"
)

df["Item_Price"] = pd.to_numeric(
    df["Item_Price"], errors="coerce"
)

# Replace missing quantity with median
df["Item_Quantity"] = df["Item_Quantity"].fillna(
    df["Item_Quantity"].median()
)

# Replace missing price with median price of the same item
df["Item_Price"] = df.groupby("Item")["Item_Price"].transform(
    lambda x: x.fillna(x.median())
)

# If item-specific median is unavailable, use overall median
df["Item_Price"] = df["Item_Price"].fillna(
    df["Item_Price"].median()
)

print("\n--- MISSING VALUES AFTER CLEANING ---")
print(df[["Item_Quantity", "Item_Price"]].isnull().sum())



--- MISSING VALUES AFTER CLEANING ---
Item_Quantity    0
Item_Price       0
dtype: int64


In [44]:
# 4. CONVERT DATETIME AND EXTRACT HOUR AND WEEKDAY
df["Order_DateTime"] = pd.to_datetime(
    df["Order_DateTime"], errors="coerce"
)

df["Order_Hour"] = df["Order_DateTime"].dt.hour

df["Weekday"] = df["Order_DateTime"].dt.day_name()

print("\n--- ORDER TIME DETAILS ---")
print(df[["Order_DateTime", "Order_Hour", "Weekday"]].head())


--- ORDER TIME DETAILS ---
       Order_DateTime  Order_Hour   Weekday
0 2026-02-15 07:36:00           7    Sunday
1 2026-02-06 18:29:00          18    Friday
2 2026-03-07 12:40:00          12  Saturday
3 2026-03-26 17:14:00          17  Thursday
4 2026-01-10 16:22:00          16  Saturday


In [45]:
# 5. TOTAL ORDERS, ITEMS SOLD AND TOTAL REVENUE
total_orders = df["Order_ID"].nunique()

total_items_sold = df["Item_Quantity"].sum()

total_revenue = (
    df["Item_Quantity"] * df["Item_Price"]
).sum()

print("\n--- BUSINESS SUMMARY ---")
print("Total Orders:", total_orders)
print("Total Items Sold:", total_items_sold)
print("Total Revenue: ₹", total_revenue)




--- BUSINESS SUMMARY ---
Total Orders: 420
Total Items Sold: 1802.0
Total Revenue: ₹ 240940.0


In [46]:
# 6. CALCULATE ROW-WISE ITEM REVENUE
df["Item_Revenue"] = (
    df["Item_Quantity"] * df["Item_Price"]
)

print("\n--- ITEM REVENUE ---")
print(df[["Item", "Item_Quantity", "Item_Price",
          "Item_Revenue"]].head())



--- ITEM REVENUE ---
           Item  Item_Quantity  Item_Price  Item_Revenue
0       Brownie            1.0       110.0         110.0
1       Biryani            1.0       220.0         220.0
2         Pasta            2.0       180.0         360.0
3  French Fries            1.0        80.0          80.0
4         Pizza            1.0       250.0         250.0


In [47]:
# 7. FIND TOP 10 MOST FREQUENTLY ORDERED DISHES
top_dishes = df.groupby("Item")["Item_Quantity"].sum().sort_values(
    ascending=False
).head(10)

print("\n--- TOP 10 POPULAR DISHES ---")
print(top_dishes)


--- TOP 10 POPULAR DISHES ---
Item
Biryani             138.0
Burger              133.0
Lemonade            128.0
French Fries        127.0
Margherita Pizza    125.0
Paneer Rice Bowl    125.0
Cold Drink          124.0
Garlic Bread        123.0
Cheese Burger       117.0
Sandwich            114.0
Name: Item_Quantity, dtype: float64


In [48]:
 #8. RANK MENU CATEGORIES BY TOTAL REVENUE
category_revenue = df.groupby("Category")[
    "Item_Revenue"
].sum().sort_values(ascending=False)

print("\n--- CATEGORY REVENUE RANKING ---")
print(category_revenue)



--- CATEGORY REVENUE RANKING ---
Category
Fast Food      89430.0
Main Course    74630.0
Snacks         32630.0
Beverages      22680.0
Desserts       21570.0
Name: Item_Revenue, dtype: float64


In [49]:
# 9. AVERAGE BILL VALUE BY ORDER TYPE
order_bills = df.groupby(
    ["Order_ID", "Order_Type"]
)["Item_Revenue"].sum().reset_index()

average_bill = order_bills.groupby(
    "Order_Type"
)["Item_Revenue"].mean()

print("\n--- AVERAGE BILL BY ORDER TYPE ---")
print(average_bill)



--- AVERAGE BILL BY ORDER TYPE ---
Order_Type
Delivery    597.342657
Dine-in     563.987342
Takeaway    558.067227
Name: Item_Revenue, dtype: float64


In [50]:
# 10. PEAK ORDERING HOURS AND BUSIEST WEEKDAY
hourly_orders = df.groupby("Order_Hour")[
    "Order_ID"
].nunique().sort_values(ascending=False)

weekday_orders = df.groupby("Weekday")[
    "Order_ID"
].nunique().sort_values(ascending=False)

print("\n--- ORDERS BY HOUR ---")
print(hourly_orders)

print("\nPeak Ordering Hour:", hourly_orders.idxmax())

print("\n--- ORDERS BY WEEKDAY ---")
print(weekday_orders)

print("\nBusiest Day:", weekday_orders.idxmax())


--- ORDERS BY HOUR ---
Order_Hour
13    36
18    31
8     31
16    31
23    28
9     27
20    27
22    27
7     26
12    26
14    24
11    19
17    19
21    19
19    17
15    16
10    16
Name: Order_ID, dtype: int64

Peak Ordering Hour: 13

--- ORDERS BY WEEKDAY ---
Weekday
Thursday     68
Sunday       67
Friday       65
Monday       63
Saturday     57
Tuesday      52
Wednesday    48
Name: Order_ID, dtype: int64

Busiest Day: Thursday


In [51]:
# 11. LIST AND TUPLE IMPLEMENTATION
top_dishes_list = top_dishes.index.tolist()

meal_categories = (
    "Breakfast",
    "Lunch",
    "Dinner",
    "Snacks",
    "Beverages"
)

print("\n--- TOP DISHES LIST ---")
print(top_dishes_list)

print("\n--- FIXED MEAL CATEGORIES TUPLE ---")
print(meal_categories)


--- TOP DISHES LIST ---
['Biryani', 'Burger', 'Lemonade', 'French Fries', 'Margherita Pizza', 'Paneer Rice Bowl', 'Cold Drink', 'Garlic Bread', 'Cheese Burger', 'Sandwich']

--- FIXED MEAL CATEGORIES TUPLE ---
('Breakfast', 'Lunch', 'Dinner', 'Snacks', 'Beverages')


In [52]:
# 12. SET FOR UNIQUE CUSTOMERS AND BILL FUNCTION
unique_customers = set(df["Customer_ID"].dropna().unique())

print("\n--- UNIQUE CUSTOMERS ---")
print(unique_customers)

print("Total Unique Customers:", len(unique_customers))


def calculate_order_bill(order_id):
    selected_order = df[df["Order_ID"] == order_id]

    if selected_order.empty:
        return "Order ID not found."

    total_bill = selected_order["Item_Revenue"].sum()

    return total_bill


# Example
sample_order_id = df["Order_ID"].iloc[0]

print("\n--- SELECTED ORDER BILL ---")
print("Order ID:", sample_order_id)
print("Total Bill: ₹", calculate_order_bill(sample_order_id))


--- UNIQUE CUSTOMERS ---
{'C0100', 'C0244', 'C0169', 'C0051', 'C0238', 'C0069', 'C0213', 'C0210', 'C0027', 'C0178', 'C0182', 'C0043', 'C0095', 'C0058', 'C0008', 'C0137', 'C0194', 'C0230', 'C0019', 'C0047', 'C0068', 'C0054', 'C0006', 'C0024', 'C0009', 'C0184', 'C0235', 'C0102', 'C0091', 'C0093', 'C0072', 'C0098', 'C0245', 'C0013', 'C0028', 'C0123', 'C0094', 'C0220', 'C0173', 'C0155', 'C0033', 'C0187', 'C0045', 'C0082', 'C0186', 'C0150', 'C0219', 'C0134', 'C0239', 'C0081', 'C0226', 'C0179', 'C0131', 'C0233', 'C0016', 'C0224', 'C0118', 'C0002', 'C0128', 'C0127', 'C0116', 'C0161', 'C0050', 'C0042', 'C0222', 'C0080', 'C0087', 'C0221', 'C0070', 'C0026', 'C0112', 'C0166', 'C0099', 'C0202', 'C0053', 'C0168', 'C0113', 'C0209', 'C0159', 'C0200', 'C0065', 'C0120', 'C0109', 'C0084', 'C0144', 'C0170', 'C0090', 'C0234', 'C0189', 'C0240', 'C0059', 'C0083', 'C0201', 'C0107', 'C0014', 'C0132', 'C0097', 'C0223', 'C0038', 'C0075', 'C0143', 'C0157', 'C0034', 'C0032', 'C0048', 'C0148', 'C0108', 'C0206', '

In [53]:
# 13. NUMPY BILL VALUE STATISTICS
bill_values = order_bills["Item_Revenue"].to_numpy()

mean_bill = np.mean(bill_values)

median_bill = np.median(bill_values)

maximum_bill = np.max(bill_values)

print("\n--- BILL VALUE STATISTICS ---")
print("Mean Bill: ₹", mean_bill)
print("Median Bill: ₹", median_bill)
print("Maximum Bill: ₹", maximum_bill)


--- BILL VALUE STATISTICS ---
Mean Bill: ₹ 573.6666666666666
Median Bill: ₹ 515.0
Maximum Bill: ₹ 1640.0


In [54]:
# 14. IQR OUTLIER DETECTION AND ORDER TIME CATEGORIES
Q1 = np.percentile(bill_values, 25)

Q3 = np.percentile(bill_values, 75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR

upper_limit = Q3 + 1.5 * IQR

outliers = order_bills[
    (order_bills["Item_Revenue"] < lower_limit) |
    (order_bills["Item_Revenue"] > upper_limit)
]

print("\n--- IQR OUTLIER ANALYSIS ---")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

print("\nUNUSUALLY LARGE BILLS:")
print(outliers)

# Create order-time categories

def get_time_category(hour):

    if pd.isna(hour):
        return "Unknown"

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["Order_Time_Category"] = df["Order_Hour"].apply(
    get_time_category
)

print("\n--- ORDER TIME CATEGORIES ---")
print(df[["Order_Hour", "Order_Time_Category"]].head())

print("\nORDERS BY TIME CATEGORY:")
print(df.groupby("Order_Time_Category")["Order_ID"].nunique())


--- IQR OUTLIER ANALYSIS ---
Q1: 340.0
Q3: 742.5
IQR: 402.5
Lower Limit: -263.75
Upper Limit: 1346.25

UNUSUALLY LARGE BILLS:
    Order_ID Order_Type  Item_Revenue
78    O00079   Delivery        1630.0
116   O00117   Takeaway        1490.0
135   O00136   Delivery        1510.0
155   O00156   Takeaway        1630.0
300   O00301   Delivery        1460.0
310   O00311   Takeaway        1640.0

--- ORDER TIME CATEGORIES ---
   Order_Hour Order_Time_Category
0           7             Morning
1          18             Evening
2          12           Afternoon
3          17             Evening
4          16           Afternoon

ORDERS BY TIME CATEGORY:
Order_Time_Category
Afternoon    133
Evening       94
Morning      119
Night         74
Name: Order_ID, dtype: int64


In [55]:
# 15. CREATE MENUITEM CLASS
class MenuItem:

    def __init__(self, item_name, category, price):

        self.item_name = item_name
        self.category = category
        self.price = price

    def display_item(self):

        print("Item Name:", self.item_name)
        print("Category:", self.category)
        print("Price: ₹", self.price)

# Create five menu items



menu1 = MenuItem("Burger", "Fast Food", 120)
menu2 = MenuItem("Pizza", "Fast Food", 250)
menu3 = MenuItem("Pasta", "Main Course", 180)
menu4 = MenuItem("Coffee", "Beverages", 80)
menu5 = MenuItem("Sandwich", "Snacks", 100)

print("\n--- MENU ITEM DETAILS ---")

menu1.display_item()
menu2.display_item()
menu3.display_item()
menu4.display_item()
menu5.display_item()


# FIVE MENU PLANNING RECOMMENDATIONS


print("\n--- MENU PLANNING RECOMMENDATIONS ---")

# RECOMMENDATION 1


print("1. Keep the top 10 most frequently ordered dishes "
      "available to satisfy customer demand.")

# RECOMMENDATION 2

print("2. Focus on high-revenue menu categories and improve "
      "their menu offerings.")

# RECOMMENDATION 3


print("3. Provide suitable offers during peak ordering hours "
      "to manage customer demand.")

# RECOMMENDATION 4


print("4. Review delivery, takeaway and dine-in average bills "
      "to develop suitable promotional offers.")

 # RECOMMENDATION 5


print("5. Review unusually large bills and customer ordering "
      "patterns to identify opportunities for menu improvement.")


# SAVE CLEANED DATASET

df.to_csv("cleaned_restaurant_orders.csv", index=False)

print("\nCleaned dataset saved successfully!")



with open("restaurant_analysis_summary.txt", "w") as file:

    file.write("RESTAURANT ORDERS ANALYSIS SUMMARY\n")

    file.write("----------------------------------\n")

    file




--- MENU ITEM DETAILS ---
Item Name: Burger
Category: Fast Food
Price: ₹ 120
Item Name: Pizza
Category: Fast Food
Price: ₹ 250
Item Name: Pasta
Category: Main Course
Price: ₹ 180
Item Name: Coffee
Category: Beverages
Price: ₹ 80
Item Name: Sandwich
Category: Snacks
Price: ₹ 100

--- MENU PLANNING RECOMMENDATIONS ---
1. Keep the top 10 most frequently ordered dishes available to satisfy customer demand.
2. Focus on high-revenue menu categories and improve their menu offerings.
3. Provide suitable offers during peak ordering hours to manage customer demand.
4. Review delivery, takeaway and dine-in average bills to develop suitable promotional offers.
5. Review unusually large bills and customer ordering patterns to identify opportunities for menu improvement.

Cleaned dataset saved successfully!
